In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Compilar el solver C++ (necesario en Colab; en local omitir si ya existe el binario)
import os
from pathlib import Path

# Detectar raiz del repo
def _find_root():
    for rel in [".", "..", "CPMP-Framework", "../CPMP-Framework"]:
        p = Path(os.path.abspath(rel))
        if (p / "Codigo_C_solver" / "main_cpmp.cpp").exists():
            return p
    return None

ROOT = _find_root()
assert ROOT, "No se encontro la raiz del repo"

cpp_dir = ROOT / "Codigo_C_solver"
frg_bin = cpp_dir / "frg"

if not frg_bin.exists():
    print("Compilando frg...")
    ret = os.system(
        f"g++ {cpp_dir}/Greedy.cpp {cpp_dir}/Layout.cpp {cpp_dir}/Bsg.cpp "
        f"{cpp_dir}/main_cpmp.cpp -o {frg_bin} -O3 -std=c++11"
    )
    os.system(f"chmod +x {frg_bin}")
    print("OK" if ret == 0 else "ERROR al compilar")
else:
    print(f"frg ya existe: {frg_bin}")


In [ ]:
import sys
import os
import re
import pandas as pd

# Detecta la carpeta src tanto en local (notebooks/) como en Colab
def _find_src():
    candidates = ["src", "../src", "CPMP-Framework/src", "../CPMP-Framework/src"]
    for rel in candidates:
        path = os.path.abspath(rel)
        if os.path.isdir(path) and os.path.exists(os.path.join(path, "settings.py")):
            return path
    return None

src = _find_src()
assert src, "No se encontró la carpeta src. Asegúrate de que el repo esté clonado correctamente."
sys.path.append(src)
print(f"src: {src}")

from settings import INSTANCE_FOLDER
from solvers.FRG import FRGSolver
from solvers.utils import summary

In [ ]:
solver_greedy = FRGSolver()          # greedy puro (beams=0)
solver_bsg    = FRGSolver(beams=10)  # BSG — ajustar beam width aquí

CVS_PATH = INSTANCE_FOLDER / "benchmarks" / "CVS"
H_MAP    = {3: 5, 4: 6, 5: 7, 6: 8, 10: 12}  # H_real -> H_solver (H_real + 2)

cvs_folders = sorted(
    [d for d in os.listdir(CVS_PATH) if (CVS_PATH / d).is_dir()]
)

In [ ]:
results = {}
all_g_solved, all_g_steps = [], []
all_b_solved, all_b_steps = [], []

for folder_name in cvs_folders:
    H_real, S_real = [int(x) for x in folder_name.split("-")]
    H           = H_MAP[H_real]
    max_steps   = S_real * H_real * 4
    folder_path = CVS_PATH / folder_name

    # Referencia Excel
    df_ref   = pd.read_excel(folder_path / f"Data{folder_name}.xlsx", header=0)
    ref_dict = dict(zip(df_ref.iloc[:, 0].astype(str), df_ref.iloc[:, 1]))

    # Archivos .dat en orden numerico
    dat_files = sorted(
        [f for f in os.listdir(folder_path) if f.endswith(".dat")],
        key=lambda f: int(re.search(r'-(\d+)\.dat$', f).group(1))
    )

    print(f"\n{'='*72}")
    print(f"  {folder_name}   H={H_real}  S={S_real}  H_solver={H}  max_steps={max_steps}")
    print(f"{'='*72}")
    print(f"{'Instancia':<12} {'Greedy':>8} {'BSG':>8} {'Ref':>8} {'G-Ref':>7} {'B-Ref':>7}")
    print(f"{'-'*55}")

    g_solved, g_steps = [], []
    b_solved, b_steps = [], []

    for filename in dat_files:
        filepath = str(folder_path / filename)
        inst_key = re.sub(r'^data', '', filename.replace('.dat', ''))

        gs, gn = solver_greedy.solve_from_path(filepath, H, max_steps)
        bs, bn = solver_bsg.solve_from_path(filepath, H, max_steps)

        ref_raw = ref_dict.get(inst_key, None)
        ref_val = int(ref_raw) if ref_raw is not None and ref_raw != 0 else None

        g_str   = str(gn) if gs else "NO"
        b_str   = str(bn) if bs else "NO"
        ref_str = str(ref_val) if ref_val else "NO"
        g_diff  = f"{gn - ref_val:+d}" if gs and ref_val else "-"
        b_diff  = f"{bn - ref_val:+d}" if bs and ref_val else "-"

        print(f"{inst_key:<12} {g_str:>8} {b_str:>8} {ref_str:>8} {g_diff:>7} {b_diff:>7}")

        g_solved.append(gs); g_steps.append(gn)
        b_solved.append(bs); b_steps.append(bn)

    all_g_solved.extend(g_solved); all_g_steps.extend(g_steps)
    all_b_solved.extend(b_solved); all_b_steps.extend(b_steps)

    ref_vals = [int(v) for v in ref_dict.values() if v and v != 0]

    print(f"{'-'*55}")
    print("Greedy:     ", end=""); summary(g_solved, g_steps)
    print("BSG:        ", end=""); summary(b_solved, b_steps)
    if ref_vals:
        print(f"Referencia:  {len(ref_vals)}/40 resueltas  avg={sum(ref_vals)/len(ref_vals):.2f}")

    results[folder_name] = {
        "H": H_real, "S": S_real,
        "g_solved": g_solved, "g_steps": g_steps,
        "b_solved": b_solved, "b_steps": b_steps,
        "ref": ref_dict,
    }

In [ ]:
print("\n" + "="*55)
print("RESUMEN GLOBAL")
print("="*55)
print("Greedy:     ", end=""); summary(all_g_solved, all_g_steps)
print("BSG:        ", end=""); summary(all_b_solved, all_b_steps)

all_ref = [int(v) for r in results.values() for v in r['ref'].values() if v and v != 0]
print(f"Referencia:  {len(all_ref)}/840 resueltas  avg={sum(all_ref)/len(all_ref):.2f}")

## Experimento: Exploración de vecinos + greedy

En cada paso se generan todos los estados vecinos (un movimiento posible),
se evalúa cada uno corriendo el greedy completo desde ahí, y se aplica el
movimiento que conduce al menor costo futuro.

**Advertencia:** es mucho más lento que greedy/BSG porque cada evaluación
es una llamada subprocess al C++. Se prueba solo en instancias pequeñas (3-3).

In [ ]:
import copy
import tempfile
from cpmp.layout import read_file

def get_feasible_moves(layout):
    """Retorna lista de (i, j) con todos los movimientos factibles."""
    moves = []
    for i in range(len(layout.stacks)):
        if len(layout.stacks[i]) == 0:
            continue
        for j in range(len(layout.stacks)):
            if i != j and len(layout.stacks[j]) < layout.H:
                moves.append((i, j))
    return moves

def neighbor_solver(instance_path, H, max_steps_outer, max_steps_greedy, solver):
    """
    Solver por exploración de vecinos:
    - En cada paso genera todos los vecinos (1 movimiento)
    - Evalúa cada vecino con el greedy completo (solver.solve_from_path)
    - Aplica el movimiento que lleva al menor costo futuro
    """
    layout = read_file(instance_path, H)
    total_moves = 0

    for _ in range(max_steps_outer):
        if layout.is_sorted():
            return True, total_moves

        moves = get_feasible_moves(layout)
        if not moves:
            break

        best_cost = float('inf')
        best_move = moves[0]  # fallback: primer movimiento si todos dan inf

        for i, j in moves:
            neighbor = copy.deepcopy(layout)
            neighbor.move(i, j)

            if neighbor.is_sorted():
                # Movimiento que resuelve directamente
                best_cost = 0
                best_move = (i, j)
                break

            # Escribir vecino a fichero temporal y evaluar con greedy
            with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as tmp:
                tmp_path = tmp.name
            try:
                solver.lay2file(neighbor, tmp_path)
                solved, cost = solver.solve_from_path(tmp_path, H, max_steps_greedy)
                if not solved:
                    cost = float('inf')
            finally:
                if os.path.exists(tmp_path):
                    os.remove(tmp_path)

            if cost < best_cost:
                best_cost = cost
                best_move = (i, j)

        if best_move is None:
            break

        layout.move(*best_move)
        total_moves += 1

    return layout.is_sorted(), total_moves

In [ ]:
# Configura aqui las carpetas a correr (por defecto todas).
# Para probar rapido cambia a: folders_to_run = ["3-3", "3-4"]
folders_to_run   = cvs_folders
max_steps_greedy = 50   # pasos max del greedy evaluador por vecino

nv_all_solved, nv_all_steps = [], []

for folder_name in folders_to_run:
    H_real, S_real  = [int(x) for x in folder_name.split("-")]
    H               = H_MAP[H_real]
    max_steps_outer = S_real * H_real * 4
    folder_path     = CVS_PATH / folder_name

    dat_files = sorted(
        [f for f in os.listdir(folder_path) if f.endswith(".dat")],
        key=lambda f: int(re.search(r'-(\d+)\.dat$', f).group(1))
    )

    print(f"
{'='*65}")
    print(f"  {folder_name}   H_solver={H}  max_steps_outer={max_steps_outer}  max_steps_greedy={max_steps_greedy}")
    print(f"{'='*65}")
    print(f"{'Instancia':<12} {'Vecinos':>8} {'Greedy':>8} {'BSG':>8}")
    print("-" * 42)

    nv_solved, nv_steps = [], []

    for filename in dat_files:
        filepath = str(folder_path / filename)
        inst_key = re.sub(r'^data', '', filename.replace('.dat', ''))

        nv_ok, nv_n = neighbor_solver(filepath, H, max_steps_outer, max_steps_greedy, solver_greedy)
        g_ok,  g_n  = solver_greedy.solve_from_path(filepath, H, max_steps_outer)
        b_ok,  b_n  = solver_bsg.solve_from_path(filepath, H, max_steps_outer)

        nv_solved.append(nv_ok); nv_steps.append(nv_n)

        nv_str = str(nv_n) if nv_ok else "NO"
        g_str  = str(g_n)  if g_ok  else "NO"
        b_str  = str(b_n)  if b_ok  else "NO"
        print(f"{inst_key:<12} {nv_str:>8} {g_str:>8} {b_str:>8}")

    nv_all_solved.extend(nv_solved); nv_all_steps.extend(nv_steps)

    print("-" * 42)
    n_ok = sum(nv_solved)
    avg  = sum(s for ok, s in zip(nv_solved, nv_steps) if ok) / n_ok if n_ok else float("inf")
    print(f"Vecinos:  {n_ok}/{len(nv_solved)} resueltas  avg={avg:.2f}" if n_ok else f"Vecinos:  {n_ok}/{len(nv_solved)} resueltas")

print("
" + "="*42)
print("RESUMEN GLOBAL — Vecinos + Greedy")
print("="*42)
n_ok = sum(nv_all_solved)
avg  = sum(s for ok, s in zip(nv_all_solved, nv_all_steps) if ok) / n_ok if n_ok else float("inf")
print(f"Vecinos:  {n_ok}/{len(nv_all_solved)} resueltas  avg={avg:.2f}" if n_ok else f"Vecinos:  {n_ok}/{len(nv_all_solved)} resueltas")
